# FMP API 직접 조회 — 최신 발표 기준 T0 vs T-4 YoY 스크리너

## 왜 이 방식인가
- 기존 스크리너는 **DB + parquet 캐시** 기반. DB 마지막 적재가 2026-08-02 라서
  그 이후 발표분(예: DY 의 2026-08-01 분기, 8월 말 발표)이 아예 없었음.
- 조회 노트북(`fmp_financial_data_lookup`)은 DB 가 아니라 **FMP API 실시간 호출**이라 최신이 보였던 것.
- 이 노트북은 DB/캐시를 완전히 우회하고 **종목별로 API 에서 직접** 최근 6개 분기 손익계산서를 받아:
  - **T0** = 가장 최근 발표 분기 (분기 라벨·달력분기 무관, 발표 순서 기준)
  - **T-4** = 4개 전 분기 -> `T0 vs T-4` = 최신 YoY
  - **T-1 vs T-5** = 직전 분기 YoY -> 최신 분기에 YoY 가 **갑자기 급등**한 기업 판별 (`accel = YoY_T0 - YoY_T1`)
- `reportedCurrency != USD` 종목(YPF ARS, TKC TRY 등)은 자동 제외 -> 통화 아티팩트 원천 차단
- `MAX_STALE_DAYS` 초과(상장폐지·피인수로 발표가 멈춘 종목) 자동 제외

주의: 전 종목 API 조회는 요금제 rate limit 내에서 수 분 소요.

In [1]:
# ---- Cell 1. 환경 & 설정 ----
import sys, os, time, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

_CANDIDATE_ROOTS = [
    r'C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast',
    r'C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy',
]
def _setup_path():
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / 'DATA').is_dir():
            if str(p) not in sys.path: sys.path.insert(0, str(p))
            return str(p)
    for c in _CANDIDATE_ROOTS:
        if os.path.isdir(os.path.join(c, 'DATA')):
            if c not in sys.path: sys.path.insert(0, c)
            return c
    raise EnvironmentError('DATA 폴더를 찾을 수 없습니다.')
_ROOT = _setup_path()
print('[PATH]', _ROOT)

import requests
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from DATA.us_target_ticker_list_2000 import ticker_list as TICKERS

# ================= 여기만 수정 =================
API_KEY        = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
TOP_N          = 100
UNIT           = 1e6        # 표시 단위 ($M)
MIN_BASE_REV   = 1e7        # 매출 T-4 하한 (USD, 1e7 = $10M)
MIN_BASE_OP    = 5e6        # 영업이익 T-4 절대값 하한
MAX_STALE_DAYS = 120        # T0 발표일이 이보다 오래되면 제외
USD_ONLY       = True       # 보고통화 USD 만
MAX_WORKERS    = 6          # 동시 요청 수 (요금제에 맞게 조절)
N_Q            = 6          # T-5 까지 필요하므로 6개 분기
SAVE_XLSX      = Path('rev_growth_rank_latest.xlsx')
# ==============================================
print(f'[설정] tickers={len(TICKERS)}, N_Q={N_Q}, workers={MAX_WORKERS}')

[PATH] C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[설정] tickers=2000, N_Q=6, workers=6


## Cell 2. API 조회 함수 (종목별 최근 6개 분기 손익계산서)

In [2]:
FMP_BASE  = 'https://financialmodelingprep.com/api/v3'
MAX_RETRY = 3

def fetch_income(ticker, n=N_Q):
    url = f'{FMP_BASE}/income-statement/{ticker}'
    for k in range(MAX_RETRY):
        try:
            r = requests.get(url, params={'apikey': API_KEY, 'period': 'quarter', 'limit': n}, timeout=30)
            if r.status_code == 429:
                time.sleep(1.5 + k); continue
            r.raise_for_status()
            data = r.json()
            if isinstance(data, dict) and 'Error Message' in data:
                raise ValueError(data['Error Message'])
            if not isinstance(data, list) or not data:
                return ticker, None
            df = pd.DataFrame(data)[['symbol', 'date', 'reportedCurrency', 'revenue', 'grossProfit', 'operatingIncome', 'netIncome', 'incomeBeforeTax', 'incomeTaxExpense', 'weightedAverageShsOut']]
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            return ticker, df.sort_values('date').reset_index(drop=True)
        except Exception:
            if k == MAX_RETRY - 1:
                return ticker, None
            time.sleep(0.5 + k * 0.5)
    return ticker, None

def fetch_all(tickers, workers=MAX_WORKERS):
    frames, fail = {}, []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fetch_income, t): t for t in tickers}
        for i, fut in enumerate(as_completed(futs), 1):
            tkr, df = fut.result()
            if df is None:
                fail.append(tkr)
            else:
                frames[tkr] = df
            if i % 100 == 0 or i == len(tickers):
                print(f'  {i}/{len(tickers)}  ok={len(frames)} fail={len(fail)}  ({time.time()-t0:,.0f}s)')
    return frames, fail

## Cell 3. 수집 실행

In [3]:
frames, fail = fetch_all(TICKERS)
print(f'수집 완료: {len(frames)}개 / 실패 {len(fail)}개')
if fail[:20]: print('실패 예:', fail[:20])

  100/2000  ok=100 fail=0  (14s)
  200/2000  ok=200 fail=0  (27s)
  300/2000  ok=300 fail=0  (40s)
  400/2000  ok=400 fail=0  (54s)
  500/2000  ok=488 fail=12  (86s)
  600/2000  ok=588 fail=12  (99s)
  700/2000  ok=688 fail=12  (112s)
  800/2000  ok=776 fail=24  (145s)
  900/2000  ok=876 fail=24  (159s)
  1000/2000  ok=976 fail=24  (172s)
  1100/2000  ok=1064 fail=36  (204s)
  1200/2000  ok=1163 fail=37  (217s)
  1300/2000  ok=1262 fail=38  (230s)
  1400/2000  ok=1351 fail=49  (262s)
  1500/2000  ok=1449 fail=51  (275s)
  1600/2000  ok=1547 fail=53  (289s)
  1700/2000  ok=1645 fail=55  (302s)
  1800/2000  ok=1732 fail=68  (333s)
  1900/2000  ok=1832 fail=68  (347s)
  2000/2000  ok=1931 fail=69  (360s)
수집 완료: 1931개 / 실패 69개
실패 예: ['GPN', 'PNR', 'ENTG', 'XPO', 'WST', 'CRS', 'ZBH', 'NWS', 'ABMD', 'HUBS', 'COG', 'RS', 'SSD', 'FR', 'MASI', 'SMTC', 'INGR', 'MHK', 'MIDD', 'FSV']


## Cell 4. T0 / T-1 / T-4 / T-5 스냅샷 구축
- T0 = 종목별 가장 최근 발표 분기 (발표 순서 기준, 달력분기 무관)
- `skip` 테이블에서 제외 사유 확인 가능 (통화, 발표 중단, 분기 부족)

In [4]:
# ---- 스냅샷 / 랭킹 함수 정의 ----
import numpy as np
import pandas as pd

REQ_COLS = ['symbol', 'date', 'reportedCurrency', 'revenue', 'operatingIncome']

def build_snapshot(frames, usd_only=True, max_stale_days=120, today=None):
    """
    frames : {ticker: income-statement DataFrame (date 오름차순, 최소 REQ_COLS 포함)}
    반환   : 종목별 T0/T-1/T-4/T-5 매출·영업이익 + YoY 테이블
    T0 = 가장 최근 발표 분기(발표일 기준), T-k = 그로부터 k개 전 분기 (분기 라벨 무관, 순서 기준)
    """
    today = pd.Timestamp(today) if today is not None else pd.Timestamp.today().normalize()
    rows, skipped = [], []
    for tkr, df in frames.items():
        if df is None or len(df) == 0:
            skipped.append((tkr, 'no_data')); continue
        df = df.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
        if len(df) < 6:
            skipped.append((tkr, f'quarters<{6} ({len(df)})')); continue
        cur = str(df['reportedCurrency'].iloc[-1]).upper()
        if usd_only and cur != 'USD':
            skipped.append((tkr, f'currency={cur}')); continue
        last = df.iloc[-1]
        stale = (today - pd.Timestamp(last['date'])).days
        if stale > max_stale_days:
            skipped.append((tkr, f'stale {stale}d')); continue
        g = lambda k, col: df.iloc[-1 - k][col]   # k=0 -> T0, k=4 -> T-4
        rows.append({
            'ticker': tkr,
            'T0_date':  pd.Timestamp(g(0, 'date')).date(),
            'T-4_date': pd.Timestamp(g(4, 'date')).date(),
            'currency': cur,
            'rev_T0':  g(0, 'revenue'),  'rev_T-1': g(1, 'revenue'),
            'rev_T-4': g(4, 'revenue'),  'rev_T-5': g(5, 'revenue'),
            'op_T0':   g(0, 'operatingIncome'), 'op_T-1': g(1, 'operatingIncome'),
            'op_T-4':  g(4, 'operatingIncome'), 'op_T-5': g(5, 'operatingIncome'),
        })
    snap = pd.DataFrame(rows)
    skip = pd.DataFrame(skipped, columns=['ticker', 'reason'])
    if snap.empty:
        return snap, skip
    def yoy(t, b):
        t, b = snap[t].astype(float), snap[b].astype(float)
        return np.where(b.abs() > 0, (t - b) / b.abs() * 100, np.nan)
    snap['rev_yoy_T0_%'] = yoy('rev_T0', 'rev_T-4')   # 최신 분기 YoY
    snap['rev_yoy_T1_%'] = yoy('rev_T-1', 'rev_T-5')  # 직전 분기 YoY
    snap['op_yoy_T0_%']  = yoy('op_T0', 'op_T-4')
    snap['op_yoy_T1_%']  = yoy('op_T-1', 'op_T-5')
    snap['rev_yoy_accel_%p'] = snap['rev_yoy_T0_%'] - snap['rev_yoy_T1_%']  # 급등 판별
    snap['op_yoy_accel_%p']  = snap['op_yoy_T0_%']  - snap['op_yoy_T1_%']
    return snap, skip

def rank_table(snap, metric='rev', n=100, unit=1e6, min_base=None, sort_by=None):
    """metric: 'rev' | 'op'.  min_base: T-4 절대값 하한(USD 원단위)."""
    b, t = f'{metric}_T-4', f'{metric}_T0'
    out = snap.copy()
    if min_base is not None:
        out = out[out[b].abs() >= min_base]
    sort_by = sort_by or f'{metric}_yoy_T0_%'
    cols = ['ticker', 'T0_date', 'T-4_date',
            f'{metric}_T-4', f'{metric}_T-1', f'{metric}_T0', f'{metric}_T-5',
            f'{metric}_yoy_T0_%', f'{metric}_yoy_T1_%', f'{metric}_yoy_accel_%p']
    out = out[cols].sort_values(sort_by, ascending=False).head(n).reset_index(drop=True)
    for c in out.columns:
        if c.startswith(metric + '_T'):
            out[c] = out[c] / unit
    return out


In [5]:
snap, skip = build_snapshot(frames, usd_only=USD_ONLY, max_stale_days=MAX_STALE_DAYS)
print(f'대상 {len(snap)}개 종목 | 제외 {len(skip)}개')
print(skip['reason'].str.split(' ').str[0].value_counts())
snap.head()

대상 1526개 종목 | 제외 405개
reason
stale           263
currency=CAD     39
currency=EUR     21
currency=CNY     15
currency=BRL     12
currency=MXN      9
currency=GBP      6
currency=KRW      5
currency=TWD      5
currency=JPY      5
currency=ZAR      4
currency=ARS      4
currency=INR      3
currency=RUB      3
currency=PHP      1
currency=PEN      1
currency=AUD      1
quarters<6        1
currency=CLP      1
currency=COP      1
currency=TRY      1
currency=IDR      1
currency=SEK      1
currency=DKK      1
currency=ILS      1
Name: count, dtype: int64


,ticker,T0_date,T-4_date,currency,rev_T0,rev_T-1,rev_T-4,rev_T-5,op_T0,op_T-1,op_T-4,op_T-5,rev_yoy_T0_%,rev_yoy_T1_%,op_yoy_T0_%,op_yoy_T1_%,rev_yoy_accel_%p,op_yoy_accel_%p
0,MSFT,2026-06-30,2025-06-30,USD,90007000000,82886000000,76441000000,70066000000,40603000000,38398000000,34323000000,32000000000,17.747021,18.297034,18.296769,19.993750,-0.550014,-1.696981
1,GOOG,2026-06-30,2025-06-30,USD,119796000000,109896000000,96428000000,90234000000,40770000000,39696000000,31271000000,30606000000,24.233625,21.790013,30.376387,29.700059,2.443612,0.676328
2,NVDA,2026-07-26,2025-07-27,USD,96221000000,81615000000,46743000000,44062000000,63734000000,53536000000,28440000000,21638000000,105.851143,85.227634,124.099859,147.416582,20.623510,-23.316723
3,AMZN,2026-06-30,2025-06-30,USD,200606000000,181519000000,167702000000,155667000000,27461000000,23852000000,19171000000,18405000000,19.620517,16.607245,43.242397,29.595219,3.013272,13.647179
4,AAPL,2026-06-27,2025-06-28,USD,109417000000,111184000000,94036000000,95359000000,35695000000,35885000000,28202000000,29589000000,16.356502,16.595182,26.569038,21.278178,-0.238681,5.290860


## Cell 5. 매출 YoY 상위 N (T0 vs T-4)

In [6]:
rev_rank = rank_table(snap, 'rev', n=TOP_N, unit=UNIT, min_base=MIN_BASE_REV)
rev_rank

,ticker,T0_date,T-4_date,rev_T-4,rev_T-1,rev_T0,rev_T-5,rev_yoy_T0_%,rev_yoy_T1_%,rev_yoy_accel_%p
0,MDC,2026-06-30,2025-06-30,32.800,1209.068,1339.630,24.772,3984.237805,4780.784757,-796.546952
1,BRKS,2026-06-30,2025-06-30,-17.418,144.795,161.178,136.939,1025.353083,5.736861,1019.616222
2,RC,2026-06-30,2025-06-30,-12.016,45.131,69.466,-74.096,678.112517,160.908821,517.203696
3,ORC,2026-06-30,2025-06-30,-28.582,157.877,164.187,21.348,674.441956,639.540004,34.901953
4,SNDK,2026-07-03,2025-06-27,1901.000,5950.000,8965.000,1695.000,371.593898,251.032448,120.561450
...,...,...,...,...,...,...,...,...,...,...
95,TNK,2026-06-30,2025-06-30,232.866,286.094,379.508,231.639,62.972697,23.508563,39.464134
96,CLS,2026-06-30,2025-06-30,2893.400,4047.000,4698.600,2648.600,62.390268,52.797704,9.592563
97,CECE,2026-06-30,2025-06-30,185.391,189.966,300.914,176.697,62.313165,7.509465,54.803700
98,LSCC,2026-07-04,2025-06-28,123.971,170.897,201.079,120.150,62.198417,42.236371,19.962046


## Cell 6. 영업이익 YoY 상위 N (T0 vs T-4)

In [7]:
op_rank = rank_table(snap, 'op', n=TOP_N, unit=UNIT, min_base=MIN_BASE_OP)
op_rank

,ticker,T0_date,T-4_date,op_T-4,op_T-1,op_T0,op_T-5,op_yoy_T0_%,op_yoy_T1_%,op_yoy_accel_%p
0,SNDK,2026-07-03,2025-06-27,18.000,4111.000000,7011.000000,-42.000000,38850.000000,9888.095238,28961.904762
1,CRH,2026-06-30,2024-03-31,18.610,2081.000000,2079.000000,1685.203141,11071.413219,23.486596,11047.926622
2,PBF,2026-06-30,2025-06-30,43.000,393.900000,1272.100000,-511.200000,2858.372093,177.053991,2681.318102
3,PMT,2026-06-30,2025-06-30,17.006,367.944000,409.629000,-6.299000,2308.732212,5941.308144,-3632.575932
4,BG,2026-06-30,2025-06-30,56.000,235.000000,1075.000000,228.000000,1819.642857,3.070175,1816.572682
...,...,...,...,...,...,...,...,...,...,...
95,SB,2026-06-30,2025-06-30,10.492,26.472999,35.910999,14.916000,242.270292,77.480551,164.789741
96,VSH,2026-07-04,2025-06-28,22.118,22.124000,75.650000,0.815000,242.029117,2614.601227,-2372.572110
97,GLOB,2026-06-30,2025-06-30,5.858,55.492000,19.855000,49.856000,238.938204,11.304557,227.633647
98,DHT,2026-06-30,2025-06-30,60.312,107.663000,203.231000,48.895000,236.966110,120.192249,116.773861


## Cell 7. 최신 분기 YoY 급등 기업 (T-1/T-5 대비 가속)
`accel = YoY(T0 vs T-4) - YoY(T-1 vs T-5)` 가 큰 순서.
직전 분기까지 평범하다가 최신 분기에 실적이 튄 기업을 잡는다.

In [8]:
accel = snap[(snap['rev_T-4'].abs() >= MIN_BASE_REV) & (snap['rev_T-5'].abs() >= MIN_BASE_REV)].copy()
accel_rank = (accel.sort_values('rev_yoy_accel_%p', ascending=False)
    [['ticker', 'T0_date', 'rev_T-5', 'rev_T-1', 'rev_T-4', 'rev_T0',
      'rev_yoy_T1_%', 'rev_yoy_T0_%', 'rev_yoy_accel_%p',
      'op_yoy_T1_%', 'op_yoy_T0_%', 'op_yoy_accel_%p']]
    .head(TOP_N).reset_index(drop=True))
for c in ['rev_T-5', 'rev_T-1', 'rev_T-4', 'rev_T0']:
    accel_rank[c] = accel_rank[c] / UNIT
accel_rank

,ticker,T0_date,rev_T-5,rev_T-1,rev_T-4,rev_T0,rev_yoy_T1_%,rev_yoy_T0_%,rev_yoy_accel_%p,op_yoy_T1_%,op_yoy_T0_%,op_yoy_accel_%p
0,BRKS,2026-06-30,136.939,144.795,-17.418,161.178,5.736861,1025.353083,1019.616222,-602.469081,92.370622,694.839703
1,RC,2026-06-30,-74.096,45.131,-12.016,69.466,160.908821,678.112517,517.203696,NaN,52.813953,NaN
2,HHC,2026-06-30,199.328,76.434,260.880,1009.186,-61.654158,286.839160,348.493318,20.517985,284.425612,263.907627
3,HHH,2026-06-30,199.328,235.917,260.880,1122.327,18.356177,330.208142,311.851965,5.729188,253.080743,247.351555
4,ARWR,2026-06-30,542.709,73.737,27.767,75.253,-86.413161,171.015954,257.429115,-137.056469,-2.744186,134.312283
...,...,...,...,...,...,...,...,...,...,...,...,...
95,ORC,2026-06-30,21.348,157.877,-28.582,164.187,639.540004,674.441956,34.901953,NaN,NaN,NaN
96,MSGS,2026-06-30,424.197,432.199,203.957,278.745,1.886388,36.668513,34.782126,-90.084101,236.897590,326.981692
97,MTDR,2026-06-30,1006.173,941.604,925.678,1186.392,-6.417286,28.164653,34.581939,-18.595095,80.961665,99.556761
98,RYI,2026-06-30,1135.700,1566.100,1169.300,2006.200,37.897332,71.572736,33.675404,908.695652,493.103448,-415.592204


## Cell 8. 결과 저장 (xlsx)

In [9]:
with pd.ExcelWriter(SAVE_XLSX, engine='openpyxl') as w:
    rev_rank.to_excel(w, sheet_name='rev_yoy_T0', index=False)
    op_rank.to_excel(w, sheet_name='op_yoy_T0', index=False)
    accel_rank.to_excel(w, sheet_name='yoy_accel', index=False)
    snap.to_excel(w, sheet_name='snapshot_all', index=False)
    skip.to_excel(w, sheet_name='excluded', index=False)
print('saved:', SAVE_XLSX.resolve())

saved: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis_FMP\rev_growth_rank_latest.xlsx


## Cell 9. 단일 종목 검증 (예: DY)

In [10]:
CHECK = 'DY'
if CHECK in frames:
    display(frames[CHECK])
    print(snap[snap['ticker'] == CHECK].T)
else:
    print(CHECK, '수집 실패 또는 제외:', skip[skip['ticker'] == CHECK])

,symbol,date,reportedCurrency,revenue,grossProfit,operatingIncome,netIncome,incomeBeforeTax,incomeTaxExpense,weightedAverageShsOut
0,DY,2025-04-26,USD,1258608000,189107000,85381000,61048000,78600000,17552000,28930000
1,DY,2025-07-26,USD,1377944000,246640000,139846000,97483000,131118000,33635000,28942000
2,DY,2025-10-25,USD,1451798000,257981000,150746000,106365000,140262000,33897000,28953396
3,DY,2026-01-31,USD,1457562000,446389000,318660000,16293000,17895000,1602000,29055000
4,DY,2026-05-02,USD,1964782000,275083000,143754000,91289000,106709000,15420000,29972366
5,DY,2026-08-01,USD,2005907000,324926000,191995000,115637000,153976000,38339000,30083100


                         505
ticker                    DY
T0_date           2026-08-01
T-4_date          2025-07-26
currency                 USD
rev_T0            2005907000
rev_T-1           1964782000
rev_T-4           1377944000
rev_T-5           1258608000
op_T0              191995000
op_T-1             143754000
op_T-4             139846000
op_T-5              85381000
rev_yoy_T0_%       45.572462
rev_yoy_T1_%       56.107541
op_yoy_T0_%        37.290305
op_yoy_T1_%         68.36767
rev_yoy_accel_%p  -10.535079
op_yoy_accel_%p   -31.077365


## Cell 10. M&A / 비유기적 성장 판별 (2단계 검증)
YoY 스크리닝의 최대 함정은 인수로 만든 성장. FMP 데이터의 3중 신호로 기계적으로 걸러낸다.

| 신호 | 데이터 | 기준(기본값) | 잡는 것 |
|---|---|---|---|
| 인수 지출 | 현금흐름표 `acquisitionsNet` 최근 4Q 합 | TTM 매출의 3% 초과 | 현금 인수 |
| 영업권 증가 | 재무상태표 `goodwill` T0 vs 4Q전 | 총자산의 2%p 초과 | 인수 프리미엄 계상 |
| 주식수 희석 | 손익계산서 `weightedAverageShsOut` | 4Q간 5% 초과 증가 | 주식교환 인수·대규모 증자 |

랭킹 상위 후보(매출·이익 상위 150 + 가속 상위)만 추가 조회하므로 API 비용은 후보 x 2콜.

In [11]:

# ---- M&A / 비유기적 성장 판별 ----
import numpy as np
import pandas as pd

def ma_flags(income_df, cashflow_df, balance_df,
             acq_rev_th=0.03, gw_asset_th=0.02, dilution_th=0.05):
    """
    티커 1개에 대한 M&A 의심 신호 3종.
    income_df   : date, revenue, weightedAverageShsOut (오름차순)
    cashflow_df : date, acquisitionsNet
    balance_df  : date, goodwill, totalAssets
    반환: dict(acq_ratio, gw_chg_ratio, dilution, flags[list], is_ma[bool])
    """
    out = {'acq_ratio': np.nan, 'gw_chg_ratio': np.nan, 'dilution': np.nan,
           'flags': [], 'is_ma': False}
    # 1) 최근 4개 분기 인수 지출 / TTM 매출
    try:
        cf = cashflow_df.dropna(subset=['date']).sort_values('date').tail(4)
        inc = income_df.dropna(subset=['date']).sort_values('date')
        acq = cf['acquisitionsNet'].fillna(0).abs().sum()
        ttm_rev = inc['revenue'].tail(4).sum()
        if ttm_rev > 0:
            out['acq_ratio'] = acq / ttm_rev
            if out['acq_ratio'] > acq_rev_th:
                out['flags'].append(f"인수지출 {out['acq_ratio']*100:.1f}%/TTM매출")
    except Exception:
        pass
    # 2) 영업권 증가 (T0 vs 4분기 전) / 총자산
    try:
        b = balance_df.dropna(subset=['date']).sort_values('date')
        if len(b) >= 5 and b['totalAssets'].iloc[-1] > 0:
            gw_chg = (b['goodwill'].iloc[-1] or 0) - (b['goodwill'].iloc[-5] or 0)
            out['gw_chg_ratio'] = gw_chg / b['totalAssets'].iloc[-1]
            if out['gw_chg_ratio'] > gw_asset_th:
                out['flags'].append(f"영업권 +{out['gw_chg_ratio']*100:.1f}%/자산")
    except Exception:
        pass
    # 3) 유통주식수 희석 (T0 vs T-4)
    try:
        sh = income_df.dropna(subset=['date']).sort_values('date')['weightedAverageShsOut']
        if len(sh) >= 5 and sh.iloc[-5] and sh.iloc[-5] > 0:
            out['dilution'] = sh.iloc[-1] / sh.iloc[-5] - 1
            if out['dilution'] > dilution_th:
                out['flags'].append(f"주식수 +{out['dilution']*100:.1f}%")
    except Exception:
        pass
    out['is_ma'] = len(out['flags']) > 0
    return out


In [12]:
# ---- 후보군에 대해 현금흐름표·재무상태표 추가 수집 (자동 보정형) ----
# - 이미 수집된 티커라도 필요한 컬럼이 빠져 있으면(구버전 수집) 자동 재수집
# - fetch 실패(None)로 남은 티커도 자동 재시도
BAL_COLS = ['goodwill', 'totalAssets', 'totalDebt', 'totalStockholdersEquity', 'cashAndShortTermInvestments']
CF_COLS  = ['acquisitionsNet']
INC_NEED = {'grossProfit', 'netIncome', 'incomeBeforeTax', 'incomeTaxExpense'}

def fetch_stmt(ticker, endpoint, cols, n=N_Q):
    url = f'{FMP_BASE}/{endpoint}/{ticker}'
    for k in range(MAX_RETRY):
        try:
            r = requests.get(url, params={'apikey': API_KEY, 'period': 'quarter', 'limit': n}, timeout=30)
            if r.status_code == 429:
                time.sleep(1.5 + k); continue
            r.raise_for_status()
            data = r.json()
            if not isinstance(data, list) or not data:
                return ticker, None
            df = pd.DataFrame(data)
            keep = [c for c in cols if c in df.columns]
            df = df[['date'] + keep]
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            return ticker, df.sort_values('date').reset_index(drop=True)
        except Exception:
            if k == MAX_RETRY - 1:
                return ticker, None
            time.sleep(0.5 + k * 0.5)
    return ticker, None

CAND_N = 150
cand = set(rank_table(snap, 'rev', n=CAND_N, unit=UNIT, min_base=MIN_BASE_REV)['ticker']) \
     | set(rank_table(snap, 'op',  n=CAND_N, unit=UNIT, min_base=MIN_BASE_OP)['ticker']) \
     | set(accel_rank['ticker'])
cand = sorted(cand & set(frames))

if 'cf_frames' not in globals():  cf_frames = {}
if 'bal_frames' not in globals(): bal_frames = {}

def _needs(store, t, cols):
    df = store.get(t)
    return df is None or not set(cols) <= set(df.columns)

need_bal = [t for t in cand if _needs(bal_frames, t, BAL_COLS)]
need_cf  = [t for t in cand if _needs(cf_frames, t, CF_COLS)]
need_inc = [t for t in cand if frames.get(t) is None or not INC_NEED <= set(frames[t].columns)]
print(f'M&A/수익성 후보 {len(cand)}개 | 재수집 필요 — balance: {len(need_bal)}, cashflow: {len(need_cf)}, income: {len(need_inc)}')

jobs = [(fetch_stmt, t, 'balance-sheet-statement', BAL_COLS, 'bal') for t in need_bal] \
     + [(fetch_stmt, t, 'cash-flow-statement', CF_COLS, 'cf') for t in need_cf]
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = {ex.submit(fn, t, ep, cols): (kind, t) for fn, t, ep, cols, kind in jobs}
    if need_inc:
        futs.update({ex.submit(fetch_income, t): ('inc', t) for t in need_inc})
    for i, fut in enumerate(as_completed(futs), 1):
        kind, t = futs[fut]
        tkr, df = fut.result()
        if kind == 'bal':   bal_frames[t] = df
        elif kind == 'cf':  cf_frames[t] = df
        elif df is not None: frames[t] = df
        if i % 100 == 0 or i == len(futs):
            print(f'  {i}/{len(futs)}')

# 최종 상태 리포트 — 남은 결측은 여기서 바로 보인다
fail_bal = [t for t in cand if _needs(bal_frames, t, BAL_COLS)]
fail_cf  = [t for t in cand if _needs(cf_frames, t, CF_COLS)]
fail_inc = [t for t in cand if frames.get(t) is None or not INC_NEED <= set(frames[t].columns)]
print(f'수집 완료 | 최종 결측 — balance: {fail_bal[:10] or "없음"}, cashflow: {fail_cf[:10] or "없음"}, income: {fail_inc[:10] or "없음"}')
if fail_bal or fail_cf:
    print('결측 티커는 이 셀을 한 번 더 실행하면 해당 티커만 재시도합니다.')

M&A/수익성 후보 268개 | 재수집 필요 — balance: 268, cashflow: 268, income: 0
  100/536
  200/536
  300/536
  400/536
  500/536
  536/536
수집 완료 | 최종 결측 — balance: ['AL', 'ALB', 'ALNY', 'AMCR', 'AMD', 'ANIP', 'STM'], cashflow: ['CMBT', 'CNC', 'CODI', 'COLM', 'COP', 'CRH'], income: 없음
결측 티커는 이 셀을 한 번 더 실행하면 해당 티커만 재시도합니다.


In [13]:
# ---- 플래그 계산 및 랭킹 테이블에 병합 ----
ma_rows = []
for t in cand:
    r = ma_flags(frames[t], cf_frames.get(t), bal_frames.get(t))
    ma_rows.append({'ticker': t, 'MA의심': r['is_ma'], 'MA사유': ' / '.join(r['flags']),
                    'acq_ratio': r['acq_ratio'], 'gw_chg_ratio': r['gw_chg_ratio'], 'dilution': r['dilution']})
ma_df = pd.DataFrame(ma_rows).set_index('ticker')
print(f"M&A 의심: {int(ma_df['MA의심'].sum())} / {len(ma_df)}")
print(ma_df[ma_df['MA의심']].sort_values('acq_ratio', ascending=False)['MA사유'].head(20).to_string())

def with_ma(rank_df):
    out = rank_df.join(ma_df[['MA의심', 'MA사유']], on='ticker')
    out['MA의심'] = out['MA의심'].fillna(False)
    out['MA사유'] = out['MA사유'].fillna('')
    return out

rev_rank_v2   = with_ma(rev_rank)
op_rank_v2    = with_ma(op_rank)
accel_rank_v2 = with_ma(accel_rank)

# 유기적(클린) 성장 랭킹: M&A 플래그 제외
rev_clean = rev_rank_v2[~rev_rank_v2['MA의심']].reset_index(drop=True)
op_clean  = op_rank_v2[~op_rank_v2['MA의심']].reset_index(drop=True)
print(f"\n클린 매출 TOP: {len(rev_clean)} / 클린 이익 TOP: {len(op_clean)}")
rev_clean.head(15)

M&A 의심: 127 / 268
ticker
TDS                                  인수지출 177.8%/TTM매출
SNDA                  인수지출 177.3%/TTM매출 / 영업권 +2.1%/자산
VSEC    인수지출 155.8%/TTM매출 / 영업권 +31.4%/자산 / 주식수 +49.9%
QXO      인수지출 126.5%/TTM매출 / 영업권 +4.7%/자산 / 주식수 +35.9%
KDP                  인수지출 102.6%/TTM매출 / 영업권 +10.9%/자산
FNV                                   인수지출 98.8%/TTM매출
ROCK                  인수지출 96.9%/TTM매출 / 영업권 +19.0%/자산
LYTS      인수지출 96.3%/TTM매출 / 영업권 +13.0%/자산 / 주식수 +9.2%
JHX      인수지출 76.1%/TTM매출 / 영업권 +33.8%/자산 / 주식수 +35.0%
ITT      인수지출 74.9%/TTM매출 / 영업권 +21.5%/자산 / 주식수 +10.0%
HHH                       인수지출 70.0%/TTM매출 / 주식수 +6.6%
HALO                   인수지출 61.0%/TTM매출 / 영업권 +6.4%/자산
AGRO      인수지출 59.6%/TTM매출 / 영업권 +4.0%/자산 / 주식수 +46.0%
CECO     인수지출 58.1%/TTM매출 / 영업권 +32.5%/자산 / 주식수 +22.7%
ESI                    인수지출 54.9%/TTM매출 / 영업권 +5.1%/자산
BCRX                     인수지출 52.7%/TTM매출 / 주식수 +21.5%
REPX                                  인수지출 52.0%/TTM매출
CTRE                     인수지출 49.9%/TTM매

,ticker,T0_date,T-4_date,rev_T-4,rev_T-1,rev_T0,rev_T-5,rev_yoy_T0_%,rev_yoy_T1_%,rev_yoy_accel_%p,MA의심,MA사유
0,MDC,2026-06-30,2025-06-30,32.800,1209.0680,1339.630000,24.772,3984.237805,4780.784757,-796.546952,False,
1,SNDK,2026-07-03,2025-06-27,1901.000,5950.0000,8965.000000,1695.000,371.593898,251.032448,120.561450,False,
2,MU,2026-05-28,2025-05-29,9301.000,23860.0000,41456.000000,8053.000,345.715514,196.287098,149.428416,False,
3,HHC,2026-06-30,2025-06-30,260.880,76.4340,1009.186000,199.328,286.839160,-61.654158,348.493318,False,
4,AGIO,2026-06-30,2025-06-30,12.455,20.7460,44.745000,8.726,259.253312,137.749255,121.504057,False,
5,LOT,2026-06-30,2025-03-31,92.823,163.3400,268.099000,271.526,188.828200,-39.843698,228.671898,False,
6,TK,2026-06-30,2025-06-30,231.694,332.4675,664.935000,231.639,186.988442,43.528292,143.460150,False,
7,ARWR,2026-06-30,2025-06-30,27.767,73.7370,75.253000,542.709,171.015954,-86.413161,257.429115,False,
8,SQM,2026-06-30,2025-06-30,1042.677,1760.0000,2468.656276,1036.630,136.761363,69.780925,66.980438,False,
9,SIMO,2026-06-30,2025-06-30,198.675,342.1050,451.001000,166.492,127.004404,105.478341,21.526063,False,


## Cell 11. 결과 저장 v2 — 원본 랭킹 + M&A 플래그 + 클린 랭킹

In [14]:
SAVE_V2 = Path('rev_growth_rank_latest_v2.xlsx')
with pd.ExcelWriter(SAVE_V2, engine='openpyxl') as w:
    rev_rank_v2.to_excel(w, sheet_name='rev_yoy_T0_flagged', index=False)
    op_rank_v2.to_excel(w, sheet_name='op_yoy_T0_flagged', index=False)
    rev_clean.to_excel(w, sheet_name='rev_clean', index=False)
    op_clean.to_excel(w, sheet_name='op_clean', index=False)
    accel_rank_v2.to_excel(w, sheet_name='yoy_accel_flagged', index=False)
    ma_df.reset_index().to_excel(w, sheet_name='ma_signals', index=False)
    snap.to_excel(w, sheet_name='snapshot_all', index=False)
    skip.to_excel(w, sheet_name='excluded', index=False)
print('saved:', SAVE_V2.resolve())

saved: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis_FMP\rev_growth_rank_latest_v2.xlsx


## Cell 12. 해석 가이드
- **MA의심 = True 라고 나쁜 기업이 아님** — "이 성장률을 유기적 성장으로 읽으면 안 된다"는 뜻. CRH·OTEX처럼 인수 통합을 잘하는 기업은 별도 스토리로 유효.
- 플래그 3종 중 2개 이상 동시 발생(인수지출 + 영업권 + 희석)이면 대형 인수 확정적.
- 반대로 클린 랭킹 상단에 남는 기업(예: NVDA·MU형)이 '순수 수요 성장' — 방송에서 가장 자신 있게 말할 수 있는 명단.
- 한계: (1) 4Q 이전에 종결된 인수의 연간 lapping 은 플래그가 꺼질 수 있음(성장률에는 아직 반영) — 8Q 조회로 확장 가능, (2) 사업 매각(divestiture)은 별도 — `acquisitionsNet` 양수 방향과 매출 급감 조합으로 추가 판별 가능.

## Cell 13. 수익성 지표 — GPM / OPM / ROIC / ROA 의 YoY 증가·증가율·가속도

각 지표를 분기 시계열로 만들고, YoY 개선을 세 가지로 측정한다 (예: OPM T-4 15% -> T0 18%):

| 산출값 | 정의 | 예시 |
|---|---|---|
| `OPM_chg_%p` | T0 - T-4 (%p) | 18% - 15% = **+3%p** |
| `OPM_chg_rate_%` | 증가폭 / \|T-4\| x 100 | 3 / 15 = **+20%** |
| `OPM_accel_%p` | (T0-T-4) - (T-1-T-5) | 직전 분기의 YoY 개선폭 대비 **가속도** |

정의: GPM = 매출총이익/매출, OPM = 영업이익/매출, ROA = 순이익/총자산(분기, 비연율화),
ROIC = 영업이익 x (1-유효세율) / (총부채+자본-현금성자산). ROIC·ROA 는 재무상태표가 수집된 후보군(cand)에서만 산출.
income fetch 에 grossProfit·netIncome·세금 항목, balance fetch 에 부채·자본·현금이 추가되었으므로 **노트북을 처음부터 재실행** 필요.

In [15]:
# ---- 수익성 지표: GPM / OPM / ROIC / ROA 의 YoY 증가·증가율·가속도 ----
QUALITY_METRICS = ['GPM', 'OPM', 'ROIC', 'ROA']

def _metric_series(inc, bal):
    """티커 1개의 분기별 지표(%) 시계열. 위치는 손익계산서 분기 기준, 재무상태표는 날짜(±40일)로 정렬."""
    inc = inc.dropna(subset=['date']).sort_values('date').reset_index(drop=True)
    out = pd.DataFrame({'date': inc['date']})
    rev = inc['revenue'].astype(float).replace(0, np.nan)
    out['GPM'] = inc['grossProfit'].astype(float) / rev * 100 if 'grossProfit' in inc else np.nan
    out['OPM'] = inc['operatingIncome'].astype(float) / rev * 100
    # 유효세율 (0~50% 클립, 세전이익<=0 이면 21% 가정)
    if {'incomeTaxExpense', 'incomeBeforeTax'} <= set(inc.columns):
        ebt = inc['incomeBeforeTax'].astype(float)
        tax = (inc['incomeTaxExpense'].astype(float) / ebt).where(ebt > 0).clip(0, 0.5).fillna(0.21)
    else:
        tax = pd.Series(0.21, index=inc.index)
    out['ROA'] = np.nan; out['ROIC'] = np.nan
    if bal is not None and len(bal):
        bal = bal.dropna(subset=['date']).sort_values('date')
        b = pd.merge_asof(out[['date']], bal, on='date', direction='nearest',
                          tolerance=pd.Timedelta('40D'))
        def _col(df, name):
            # 컬럼이 없으면(구버전 수집 데이터) NaN Series 반환 — int 0 이 아니라 항상 Series 를 돌려준다
            return df[name].astype(float) if name in df.columns else pd.Series(np.nan, index=df.index, dtype=float)
        ta = _col(b, 'totalAssets').replace(0, np.nan)
        if 'netIncome' in inc.columns:
            out['ROA'] = inc['netIncome'].astype(float) / ta * 100          # 분기 기준(비연율화) — YoY 비교에는 동일 기준이면 충분
        need_ic = {'totalDebt', 'totalStockholdersEquity'}
        if need_ic <= set(b.columns):
            ic = (_col(b, 'totalDebt').fillna(0)
                  + _col(b, 'totalStockholdersEquity').fillna(0)
                  - _col(b, 'cashAndShortTermInvestments').fillna(0))
            ic = ic.where(ic > 0)
            out['ROIC'] = inc['operatingIncome'].astype(float) * (1 - tax) / ic * 100
    return out

def quality_table(frames, bal_frames, tickers):
    """
    지표별 산출 (예: OPM):
      OPM_T-4, OPM_T0, OPM_chg_%p = T0 - T-4, OPM_chg_rate_% = chg / |T-4| x 100
      OPM_accel_%p = (T0 - T-4) - (T-1 - T-5)   ... 직전 분기 YoY 개선폭 대비 가속
    """
    rows = []
    for t in tickers:
        inc = frames.get(t)
        if inc is None or len(inc) < 6:
            continue
        ms = _metric_series(inc, bal_frames.get(t))
        row = {'ticker': t}
        for m in QUALITY_METRICS:
            s = ms[m]
            v = lambda k: s.iloc[-1 - k] if len(s) > k else np.nan   # k=0 -> T0
            t0, t4, t1, t5 = v(0), v(4), v(1), v(5)
            chg = t0 - t4
            row[f'{m}_T-4'] = round(t4, 2) if pd.notna(t4) else np.nan
            row[f'{m}_T0'] = round(t0, 2) if pd.notna(t0) else np.nan
            row[f'{m}_chg_%p'] = round(chg, 2) if pd.notna(chg) else np.nan
            row[f'{m}_chg_rate_%'] = round(chg / abs(t4) * 100, 1) if pd.notna(chg) and pd.notna(t4) and abs(t4) > 1e-9 else np.nan
            row[f'{m}_accel_%p'] = round((t0 - t4) - (t1 - t5), 2) if pd.notna(t0) and pd.notna(t4) and pd.notna(t1) and pd.notna(t5) else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index('ticker')


def quality_diagnose(frames, bal_frames, tickers):
    """지표가 NaN 이 되는 원인을 티커 단위로 진단."""
    rows = []
    for t in tickers:
        inc = frames.get(t); bal = bal_frames.get(t)
        why = []
        if inc is None or len(inc) < 6: why.append('income부족')
        else:
            if 'netIncome' not in inc.columns: why.append('netIncome없음(income 재수집 필요)')
            if 'grossProfit' not in inc.columns: why.append('grossProfit없음')
        if bal is None: why.append('balance없음(fetch실패)')
        elif not {'totalDebt', 'totalStockholdersEquity'} <= set(bal.columns): why.append('부채·자본없음(balance 재수집 필요)')
        if why: rows.append({'ticker': t, '원인': ' / '.join(why)})
    d = pd.DataFrame(rows)
    if len(d):
        print(f'데이터 결측 {len(d)}개 티커 — 수집 셀(Cell 10)을 재실행하면 자동 보정됩니다.')
        print(d['원인'].value_counts().to_string())
    else:
        print('전 티커 데이터 완비 — GPM/OPM/ROIC/ROA 모두 산출 가능.')
    return d

_diag = quality_diagnose(frames, bal_frames, cand)

quality_df = quality_table(frames, bal_frames, cand)
print(f'수익성 지표 산출: {len(quality_df)}개 종목')
quality_df[['OPM_T-4', 'OPM_T0', 'OPM_chg_%p', 'OPM_chg_rate_%', 'OPM_accel_%p']].head(10)

데이터 결측 7개 티커 — 수집 셀(Cell 10)을 재실행하면 자동 보정됩니다.
원인
balance없음(fetch실패)    7
수익성 지표 산출: 268개 종목


,OPM_T-4,OPM_T0,OPM_chg_%p,OPM_chg_rate_%,OPM_accel_%p
ticker,,,,,
AAOI,-15.52,-12.88,2.63,17.0,2.28
AAON,7.57,10.99,3.42,45.2,2.84
ACTG,-24.17,7.43,31.61,130.8,77.80
AD,3.82,738.42,734.60,19225.5,729.48
ADM,0.47,3.95,3.48,737.0,2.72
AEHR,-16.61,-5.03,11.58,69.7,46.48
AEIS,7.16,16.57,9.41,131.4,3.10
AER,22.98,55.24,32.26,140.4,31.92
AG,14.91,47.54,32.63,218.8,-2.44


In [16]:
# ---- 랭킹 테이블에 수익성 지표 병합 + 마진 개선 랭킹 ----
def with_quality(rank_df, metrics=('OPM', 'ROIC', 'ROA')):
    cols = []
    for m in metrics:
        cols += [f'{m}_T-4', f'{m}_T0', f'{m}_chg_%p', f'{m}_chg_rate_%', f'{m}_accel_%p']
    return rank_df.join(quality_df[cols], on='ticker')

rev_rank_v3 = with_quality(rev_rank_v2)
op_rank_v3  = with_quality(op_rank_v2, metrics=('GPM', 'OPM', 'ROIC', 'ROA'))

# 수익성 자체의 개선 랭킹 (증익 랭킹과 별개): OPM 개선폭 + ROIC 개선 동시 충족
qual_rank = quality_df.join(ma_df[['MA의심']], how='left')
qual_rank = qual_rank[(qual_rank['OPM_T-4'] > 0)]   # 전년 흑자 마진 기업만
qual_rank = qual_rank.sort_values('OPM_chg_%p', ascending=False)
print('OPM 개선폭 TOP 15 (전년 OPM 플러스 기업):')
qual_rank[['OPM_T-4', 'OPM_T0', 'OPM_chg_%p', 'OPM_chg_rate_%', 'OPM_accel_%p',
           'ROIC_chg_%p', 'ROA_chg_%p', 'MA의심']].head(15)

OPM 개선폭 TOP 15 (전년 OPM 플러스 기업):


,OPM_T-4,OPM_T0,OPM_chg_%p,OPM_chg_rate_%,OPM_accel_%p,ROIC_chg_%p,ROA_chg_%p,MA의심
ticker,,,,,,,,
AD,3.82,738.42,734.60,19225.5,729.48,14.13,10.34,False
USM,3.82,738.42,734.60,19225.5,430.07,16.10,NaN,False
TDS,3.37,120.67,117.30,3477.9,121.52,6.37,3.47,True
ARR,37.10,143.21,106.11,286.0,78.04,-2.38,0.97,True
PMT,7.17,85.76,78.59,1096.1,-6.28,1.62,0.08,False
SNDK,0.95,78.20,77.26,8159.2,5.69,54.10,30.85,False
LPG,18.51,75.93,57.42,310.1,22.49,8.18,6.69,True
MU,23.32,80.37,57.05,244.6,11.44,31.50,18.65,False
HR,5.49,59.26,53.77,978.5,48.61,1.40,1.06,False


## Cell 14. 결과 저장 v3 — 랭킹 + M&A 플래그 + 수익성 지표

In [17]:
# ---- v3 저장: 기존 시트 + 수익성 지표 ----
SAVE_V3 = Path('rev_growth_rank_latest_v3.xlsx')
with pd.ExcelWriter(SAVE_V3, engine='openpyxl') as w:
    rev_rank_v3.to_excel(w, sheet_name='rev_yoy_flag_quality', index=False)
    op_rank_v3.to_excel(w, sheet_name='op_yoy_flag_quality', index=False)
    rev_clean.to_excel(w, sheet_name='rev_clean', index=False)
    op_clean.to_excel(w, sheet_name='op_clean', index=False)
    qual_rank.reset_index().to_excel(w, sheet_name='opm_improve_rank', index=False)
    quality_df.reset_index().to_excel(w, sheet_name='quality_all', index=False)
    accel_rank_v2.to_excel(w, sheet_name='yoy_accel_flagged', index=False)
    ma_df.reset_index().to_excel(w, sheet_name='ma_signals', index=False)
    snap.to_excel(w, sheet_name='snapshot_all', index=False)
    skip.to_excel(w, sheet_name='excluded', index=False)
print('saved:', SAVE_V3.resolve())

saved: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\FS_Data_Analysis_FMP\rev_growth_rank_latest_v3.xlsx
